# Training Notebook

This notebook demonstrates:
- Model training
- Training loop implementation
- Loss tracking
- Model checkpointing


In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from utils.dataloader import get_loaders
from models.resnet_transfer import get_model as get_resnet_model
from utils.metrics import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Load data
train_loader, val_loader, test_loader = get_loaders(batch_size=32)

# Initialize model
model = get_resnet_model(num_classes=1, pretrained=True, freeze_backbone=True)
model = model.to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Model initialized!")


In [ ]:
# Training loop (example for one epoch)
model.train()
for epoch in range(1):  # Change to desired number of epochs
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')
    
    print(f'Epoch {epoch+1}, Average Loss: {running_loss/len(train_loader):.4f}')
